In [1]:

import os
os.getcwd()
os.chdir("/gradution project")
os.getcwd()

'e:\\gradution project'

In [2]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from src.similarity_model import preprocess_dataset
from src.similarity_model import train_embedding_engine
from src.similarity_model import search_by_text
from src.similarity_model import find_similar_projects
from src.similarity_model import extract_features

from src.similarity_model import normalize_text
from src.similarity_model import compute_feature_similarity
from Data.database.sql_connector import (
    load_preprocessed_projects,
    engine
)

print("All modules imported successfully")


 CONFIG LOADED:
ENV: development
DEBUG_MODE: True
MODELS: ['gemini-3.1-flash-lite-preview', 'gemini-2.5-flash-lite', 'gemini-2.5-flash', 'gemini-2.5-pro']
MAX_RETRIES: 3
IDEA_TEMP: 0.9



2026-06-04 00:29:43,014 | INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
e:\gradution project\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-06-04 00:29:46,381 | INFO | Use pytorch device_name: cpu
2026-06-04 00:29:46,388 | INFO | Loading faiss with AVX2 support.
2026-06-04 00:29:46,418 | INFO | Successfully loaded faiss with AVX2 support.


SQL Connected Successfully
All modules imported successfully


In [33]:
from sqlalchemy import create_engine
import urllib

SERVER = "innotrack-sql-server.database.windows.net"
DATABASE = "InnoTrackDB"
USERNAME = "innotrackadmin"
PASSWORD = "Innotrack@admin233"

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

print("Engine created")

Engine created


In [4]:
with engine.connect() as conn:

    tables = pd.read_sql(
        """
        SELECT TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        """,
        conn
    )

tables

,TABLE_NAME
0,Teams
1,ChatRooms
2,ChatMessageHiddens
3,JoinRequests
4,ChatMessageReactions
5,Projects
6,TeamMembers
7,ProjectTechnologies_Backup
8,ChatMessages
9,Feedbacks


In [5]:
query = """
SELECT *
FROM PreProcessed_Projects
"""

df = pd.read_sql(query, engine)

df.head()

,id,submitted_at,project_title,student_names,year,abstract,description,problem_statement,proposed_solution,objectives,full_content,clean_text,word_count,features
0,1,NaT,3D hand game for neuromuscular patients,"Ahmed Mansour Mohamed Saber, Ahmed Mohamed Moh...",2017,In this project we have designed and implement...,A virtual rehabilitation system that uses a Le...,Neuromuscular patients suffer from nerve atrop...,The development of a 3D interactive game integ...,1. Develop a scalable and maintainable solutio...,3D hand game for neuromuscular patients. 3D ha...,3d hand game for neuromuscular patients. 3d ha...,172,"""\""[\\\""Leap Motion controller sensor\\\"", \\\..."
1,2,NaT,3D Laser Scanning,"Aya Essam Hegazi, Asmaa Abd EL-Aziz, Ebtehal E...",2024,3D scanning is used in many applications such ...,This project implements a low-cost 3D laser sc...,Existing 3D scanning devices are often extreme...,A low-cost 3D laser scanning system that utili...,1. Improve overall productivity and workflow o...,3D Laser Scanning. 3D Laser Scanning. 3D scann...,3d laser scanning. 3d laser scanning. 3d scann...,185,"""\""[\\\""3d laser scanning\\\"", \\\""Hand-held l..."
2,3,NaT,A Smart Automatic System for Criminal Identifi...,"Yousef Yacoub Mohammed, Ahmed Mohamed Hassan,\...",2020,The increasing use of biometric technologies i...,This project develops an automated criminal id...,"Traditional identification methods, such as ph...",A real-time facial recognition system develope...,1. Support future scalability and feature expa...,A Smart Automatic System for Criminal Identifi...,a smart automatic system for criminal identifi...,138,"""\""[\\\""real-time face recognition system\\\"",..."
3,4,NaT,Advanced Educational Platform “ABSTHALK”,"Mohamed Nasser Maher, Karim Ashraf Salah Eldie...",2025,The Educational Platform for Students and Teac...,"ABSTHALK is a comprehensive, role-based e-lear...",Traditional learning methods often lack access...,"The project proposes a structured, role-based,...",1. Provide interactive educational tools and r...,Advanced Educational Platform “ABSTHALK”. Adva...,advanced educational platform absthalk . advan...,192,"""\""[\\\""Role-based management system\\\"", \\\""..."
4,5,NaT,Agricultural Information and Management System,"Ahmed Mohamed, Omar Hassan, Mahmoud Ali ,Mazen...",2020,It is a permanent link between the decision-ma...,This project is an integrated information syst...,The competent authorities of the Ministry of A...,The development of an integrated information s...,1. Reduce operational complexity and improve e...,Agricultural Information and Management System...,agricultural information and management system...,109,"""\""[\\\""centralized database\\\"", \\\""track la..."


In [6]:
print(df.columns.tolist())

['id', 'submitted_at', 'project_title', 'student_names', 'year', 'abstract', 'description', 'problem_statement', 'proposed_solution', 'objectives', 'full_content', 'clean_text', 'word_count', 'features']


In [7]:
df = df.rename(columns={
    "Title": "project_title",
    "Description": "description",
    "Abstract": "abstract"
})

In [8]:
query = """
SELECT *
FROM PreProcessed_Projects
"""

clean_df = pd.read_sql(query, engine)

clean_df.head()

,id,submitted_at,project_title,student_names,year,abstract,description,problem_statement,proposed_solution,objectives,full_content,clean_text,word_count,features
0,1,NaT,3D hand game for neuromuscular patients,"Ahmed Mansour Mohamed Saber, Ahmed Mohamed Moh...",2017,In this project we have designed and implement...,A virtual rehabilitation system that uses a Le...,Neuromuscular patients suffer from nerve atrop...,The development of a 3D interactive game integ...,1. Develop a scalable and maintainable solutio...,3D hand game for neuromuscular patients. 3D ha...,3d hand game for neuromuscular patients. 3d ha...,172,"""\""[\\\""Leap Motion controller sensor\\\"", \\\..."
1,2,NaT,3D Laser Scanning,"Aya Essam Hegazi, Asmaa Abd EL-Aziz, Ebtehal E...",2024,3D scanning is used in many applications such ...,This project implements a low-cost 3D laser sc...,Existing 3D scanning devices are often extreme...,A low-cost 3D laser scanning system that utili...,1. Improve overall productivity and workflow o...,3D Laser Scanning. 3D Laser Scanning. 3D scann...,3d laser scanning. 3d laser scanning. 3d scann...,185,"""\""[\\\""3d laser scanning\\\"", \\\""Hand-held l..."
2,3,NaT,A Smart Automatic System for Criminal Identifi...,"Yousef Yacoub Mohammed, Ahmed Mohamed Hassan,\...",2020,The increasing use of biometric technologies i...,This project develops an automated criminal id...,"Traditional identification methods, such as ph...",A real-time facial recognition system develope...,1. Support future scalability and feature expa...,A Smart Automatic System for Criminal Identifi...,a smart automatic system for criminal identifi...,138,"""\""[\\\""real-time face recognition system\\\"",..."
3,4,NaT,Advanced Educational Platform “ABSTHALK”,"Mohamed Nasser Maher, Karim Ashraf Salah Eldie...",2025,The Educational Platform for Students and Teac...,"ABSTHALK is a comprehensive, role-based e-lear...",Traditional learning methods often lack access...,"The project proposes a structured, role-based,...",1. Provide interactive educational tools and r...,Advanced Educational Platform “ABSTHALK”. Adva...,advanced educational platform absthalk . advan...,192,"""\""[\\\""Role-based management system\\\"", \\\""..."
4,5,NaT,Agricultural Information and Management System,"Ahmed Mohamed, Omar Hassan, Mahmoud Ali ,Mazen...",2020,It is a permanent link between the decision-ma...,This project is an integrated information syst...,The competent authorities of the Ministry of A...,The development of an integrated information s...,1. Reduce operational complexity and improve e...,Agricultural Information and Management System...,agricultural information and management system...,109,"""\""[\\\""centralized database\\\"", \\\""track la..."


In [9]:
print(clean_df.shape)


(255, 14)


In [10]:
clean_df["features"].apply(len).describe()

count    255.000000
mean     236.031373
std       87.747619
min       24.000000
25%      173.500000
50%      225.000000
75%      287.000000
max      719.000000
Name: features, dtype: float64

In [11]:
clean_df.to_parquet("Data_gemini/projects_clean_gemini.parquet", index=False)
clean_df.to_csv("Data_gemini/projects_clean_gemini.csv", index=False)

print("Saved cleaned dataset")

Saved cleaned dataset


In [12]:
test_df = pd.read_parquet(
    "Data_gemini/projects_clean_gemini.parquet"
)

print(test_df.shape)

(255, 14)


In [13]:
print(clean_df.columns.tolist())

['id', 'submitted_at', 'project_title', 'student_names', 'year', 'abstract', 'description', 'problem_statement', 'proposed_solution', 'objectives', 'full_content', 'clean_text', 'word_count', 'features']


In [14]:
test_df = pd.read_sql(
    "SELECT TOP 5 * FROM PreProcessed_Projects",
    engine
)

test_df.head()

,id,submitted_at,project_title,student_names,year,abstract,description,problem_statement,proposed_solution,objectives,full_content,clean_text,word_count,features
0,1,None,3D hand game for neuromuscular patients,"Ahmed Mansour Mohamed Saber, Ahmed Mohamed Moh...",2017,In this project we have designed and implement...,A virtual rehabilitation system that uses a Le...,Neuromuscular patients suffer from nerve atrop...,The development of a 3D interactive game integ...,1. Develop a scalable and maintainable solutio...,3D hand game for neuromuscular patients. 3D ha...,3d hand game for neuromuscular patients. 3d ha...,172,"""\""[\\\""Leap Motion controller sensor\\\"", \\\..."
1,2,None,3D Laser Scanning,"Aya Essam Hegazi, Asmaa Abd EL-Aziz, Ebtehal E...",2024,3D scanning is used in many applications such ...,This project implements a low-cost 3D laser sc...,Existing 3D scanning devices are often extreme...,A low-cost 3D laser scanning system that utili...,1. Improve overall productivity and workflow o...,3D Laser Scanning. 3D Laser Scanning. 3D scann...,3d laser scanning. 3d laser scanning. 3d scann...,185,"""\""[\\\""3d laser scanning\\\"", \\\""Hand-held l..."
2,3,None,A Smart Automatic System for Criminal Identifi...,"Yousef Yacoub Mohammed, Ahmed Mohamed Hassan,\...",2020,The increasing use of biometric technologies i...,This project develops an automated criminal id...,"Traditional identification methods, such as ph...",A real-time facial recognition system develope...,1. Support future scalability and feature expa...,A Smart Automatic System for Criminal Identifi...,a smart automatic system for criminal identifi...,138,"""\""[\\\""real-time face recognition system\\\"",..."
3,4,None,Advanced Educational Platform “ABSTHALK”,"Mohamed Nasser Maher, Karim Ashraf Salah Eldie...",2025,The Educational Platform for Students and Teac...,"ABSTHALK is a comprehensive, role-based e-lear...",Traditional learning methods often lack access...,"The project proposes a structured, role-based,...",1. Provide interactive educational tools and r...,Advanced Educational Platform “ABSTHALK”. Adva...,advanced educational platform absthalk . advan...,192,"""\""[\\\""Role-based management system\\\"", \\\""..."
4,5,None,Agricultural Information and Management System,"Ahmed Mohamed, Omar Hassan, Mahmoud Ali ,Mazen...",2020,It is a permanent link between the decision-ma...,This project is an integrated information syst...,The competent authorities of the Ministry of A...,The development of an integrated information s...,1. Reduce operational complexity and improve e...,Agricultural Information and Management System...,agricultural information and management system...,109,"""\""[\\\""centralized database\\\"", \\\""track la..."


In [15]:
from src.similarity_model.preprocessing import (
    extract_features,
    normalize_text
)

def check_duplicates(features):

    found = False

    for i in range(len(features)):
        for j in range(i + 1, len(features)):

            a = set(features[i].split())
            b = set(features[j].split())

            overlap = len(a & b)

            if overlap > 0:
                found = True
                print(
                    f"{features[i]} <-> {features[j]} "
                    f"(shared={overlap})"
                )

    if not found:
        print("No duplicate overlaps found")


tests = {
    "Hospital Test": """
        Hospital management system with
        appointment booking,
        online appointment booking,
        patient records,
        medical records,
        doctor dashboard,
        physician dashboard,
        AI chatbot,
        intelligent chatbot
    """,

    "Machine Learning Test": """
        Machine learning system using machine learning
        for machine learning prediction and machine learning analysis.
    """,

    "Face Recognition Test": """
        Face recognition attendance system using deep learning,
        computer vision,
        real-time face detection,
        student attendance management and mobile application.
    """
}

for name, query in tests.items():

    print("=" * 80)
    print(name)
    print("=" * 80)

    features = extract_features(
        normalize_text(query)
    )

    print(f"Feature Count: {len(features)}")
    print()

    for f in features:
        print("-", f)

    print("\nDuplicate Check:")
    check_duplicates(features)

    print("\n")

Hospital Test
USING GEMINI FEATURE EXTRACTOR
CALLING GEMINI


2026-06-04 00:30:08,804 | INFO | [LLM] model=gemini-3.1-flash-lite-preview | task=feature | attempt=1
2026-06-04 00:30:08,805 | INFO | AFC is enabled with max remote calls: 10.
2026-06-04 00:30:09,875 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"


PARSED FEATURES:
['appointment booking', 'patient records management', 'medical records storage', 'doctor dashboard', 'physician dashboard', 'ai chatbot']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Feature Count: 5

- appointment booking
- patient records management
- medical records storage
- doctor dashboard
- ai chatbot

Duplicate Check:
patient records management <-> medical records storage (shared=1)


Machine Learning Test
USING GEMINI FEATURE EXTRACTOR
CALLING GEMINI


...

Feature Count: 2

- prediction
- analysis

Duplicate Check:
No duplicate overlaps found


Face Recognition Test
USING GEMINI FEATURE EXTRACTOR
CALLING GEMINI


2026-06-04 00:30:21,508 | INFO | [LLM] model=gemini-3.1-flash-lite-preview | task=feature | attempt=1
2026-06-04 00:30:21,509 | INFO | AFC is enabled with max remote calls: 10.
2026-06-04 00:30:22,145 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"


PARSED FEATURES:
['face recognition', 'real-time face detection', 'student attendance management', 'mobile application']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Feature Count: 4

- face recognition
- real-time face detection
- student attendance management
- mobile application

Duplicate Check:
face recognition <-> real-time face detection (shared=1)




In [16]:
from Data.database.sql_connector import engine

engine.dispose()

In [17]:
results = find_similar_projects(
    title="AI Clinic Management System",
    description="""
    Smart clinic management platform with
    appointment booking,
    patient records,
    doctor dashboard,
    AI chatbot.
    """,
    top_k=5
)

results[[
    "project_title",
    "semantic_score",
    "feature_score",
    "coverage",
    "hybrid_score",
    "duplicate_risk"
]]

2026-06-04 00:30:22,479 | INFO | Loading models and artifacts...
2026-06-04 00:30:22,481 | INFO | Loading model: all-MiniLM-L6-v2
2026-06-04 00:30:22,481 | INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
e:\gradution project\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-06-04 00:30:24,618 | INFO | Use pytorch device_name: cpu
2026-06-04 00:30:24,624 | INFO | Loading FAISS index...
2026-06-04 00:30:24,627 | INFO | Loading feature model: all-MiniLM-L6-v2
2026-06-04 00:30:24,628 | INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-06-04 00:30:26,763 | INFO | Use pytorch device_name: cpu
2026-06-04 00:30:26,767 | INFO | Loading metadata from Azure SQL...
2026-06-04 00:30:32,815 | INFO | Preparing query...


USING GEMINI FEATURE EXTRACTOR
CALLING GEMINI


2026-06-04 00:30:36,816 | INFO | [LLM] model=gemini-3.1-flash-lite-preview | task=feature | attempt=1
2026-06-04 00:30:36,817 | INFO | AFC is enabled with max remote calls: 10.
2026-06-04 00:30:37,822 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"


PARSED FEATURES:
['appointment booking', 'patient records', 'doctor dashboard', 'ai chatbot']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,project_title,semantic_score,feature_score,coverage,hybrid_score,duplicate_risk
0,Detecting Diseases Using Chatbot and Booking C...,0.7480,0.0,0.0,0.05,Very Low
1,Clinical Information System,0.6479,0.0,0.0,0.05,Very Low
2,Doctor 4 U,0.6437,0.0,0.0,0.05,Very Low
3,Health Care Management System,0.6402,0.0,0.0,0.05,Very Low
4,Hospital Management System,0.6397,0.0,0.0,0.05,Very Low


In [18]:
project_a = [
    "appointment booking",
    "patient records",
    "doctor dashboard",
    "ai chatbot",
    "clinic management"
]

project_b = [
    "booking doctor appointments",
    "medical records",
    "doctor dashboard",
    "intelligent chatbot",
    "hospital management"
]

result = compute_feature_similarity(
    project_a,
    project_b
)

print(result)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'score': 0.8726, 'coverage': 0.8, 'shared_count': 4, 'matches': [{'feature_a': 'appointment booking', 'feature_b': 'booking doctor appointments', 'score': 0.821}, {'feature_a': 'patient records', 'feature_b': 'medical records', 'score': 0.895}, {'feature_a': 'doctor dashboard', 'feature_b': 'doctor dashboard', 'score': 1.0}, {'feature_a': 'ai chatbot', 'feature_b': 'intelligent chatbot', 'score': 0.899}], 'unique_a': ['clinic management'], 'unique_b': ['hospital management']}


In [19]:
from src.similarity_model import compute_originality

print(
    compute_originality(
        hybrid_score=0.30,
        unique_query_features=7,
        total_query_features=8
    )
)

82.25


In [20]:
from src.similarity_model.embedding_engine import (
    train_embedding_engine
)

engine = train_embedding_engine()

print("Training Completed")

2026-06-04 00:30:41,636 | INFO | Loading processed dataset from Azure SQL...
2026-06-04 00:30:46,601 | INFO | Loading embedding model: all-MiniLM-L6-v2
2026-06-04 00:30:46,602 | INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
e:\gradution project\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-06-04 00:30:49,233 | INFO | Use pytorch device_name: cpu
2026-06-04 00:30:49,243 | INFO | Generating embeddings for 255 projects...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2026-06-04 00:31:05,278 | INFO | FAISS index built successfully with 255 vectors.
2026-06-04 00:31:05,299 | INFO | Artifacts saved to models
2026-06-04 00:31:05,301 | INFO | Embedding engine completed successfully.


Training Completed


In [21]:
from src.similarity_model.embedding_engine import ProjectEmbedder

engine = ProjectEmbedder()
engine.load_artifacts()

results = engine.search(
    "hospital management system with appointment booking and patient records",
    k=5
)

print(results)

2026-06-04 00:31:05,325 | INFO | Loading embedding model: all-MiniLM-L6-v2
2026-06-04 00:31:05,327 | INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
e:\gradution project\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-06-04 00:31:07,549 | INFO | Use pytorch device_name: cpu
2026-06-04 00:31:07,583 | INFO | Artifacts loaded successfully.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   project_id                                          title technologies  \
0         105                     Hospital Management System                
1          47                    Clinical Information System                
2         110                  Health Care Management System                
3          62                                     Doctor 4 U                
4         112  health services & medical outcomes monitoring                

   similarity_score  
0            0.8216  
1            0.6907  
2            0.6779  
3            0.5829  
4            0.5801  


In [22]:
result = compute_feature_similarity(
    [
        "machine learning system",
        "machine learning prediction",
        "machine learning analysis"
    ],
    [
        "machine learning platform",
        "predictive machine learning",
        "ml analytics"
    ]
)

print(result)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'score': 0.8866, 'coverage': 1.0, 'shared_count': 1, 'matches': [{'feature_a': 'machine learning system', 'feature_b': 'machine learning platform', 'score': 0.838}], 'unique_a': [], 'unique_b': ['ml analytics']}


In [23]:
from src.similarity_model.hybrid_ranker import (
    compute_hybrid_score
)

print(
    compute_hybrid_score(
        semantic_score=0.95,
        feature_score=0.0,
        coverage=0.0,
        feature_count=5,
        unique_query_count=5
    )
)

0.05


In [24]:
row = clean_df[
    clean_df["project_title"] == "Smart Library"
].iloc[0]

for column in clean_df.columns:
    print(f"\n{column}:")
    print(row[column])


id:
207

submitted_at:
NaT

project_title:
Smart Library

student_names:
Abdel Hamid Abdel Nasser, Mahmoud Tamer Mahmoud, Amer Saed Mohamed Ali Amer, Tahany Adel Faragallah, Hala Ahmed Saad Salem, Mohamed Khaled Mohamed

year:
2022

abstract:
Egypt is striving and our efforts are focused these days towards digital transformation and the nationalization of all its government facilities, including the higher education sector. With more than 4 million university students and up to 644, 000 graduates annually, we need smart digital systems that support the educational process and scientific research. Therefore, we have developed a smart library application that takes care of books, recommendations, and user opinions, and provides the appropriate electronic environment for university students to find and nominate appropriate books through an electronic application based on artificial intelligence. Where, using artificial intelligence algorithms, the application will analyze book data and s